# Recipe GenAI System — Embedding Generation & Evaluation

This notebook:
1. Loads the cleaned recipe dataset
2. Generates sentence-transformer embeddings
3. Builds and evaluates the FAISS vector index
4. Runs sample ingredient queries to verify retrieval quality

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('../'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE

from src.embedding.embedder import RecipeEmbedder
from src.embedding.vector_store import RecipeVectorStore
from src.utils.config import settings

## 1. Load Cleaned Data

In [ ]:
processed_path = settings.paths.processed_data / 'recipes_clean.parquet'
df = pd.read_parquet(processed_path)
print(f'Loaded {len(df):,} cleaned recipes')
df.head(2)

## 2. Initialise Embedder

In [ ]:
embedder = RecipeEmbedder()
print(f'Model: {embedder.model_name}')
print(f'Vector dimension: {embedder.vector_dim}')

## 3. Generate Embeddings

In [ ]:
import time
t0 = time.time()
vectors = embedder.encode_recipes(df, batch_size=128, show_progress=True)
elapsed = round(time.time() - t0, 1)

print(f'Embedding complete in {elapsed}s')
print(f'Matrix shape: {vectors.shape}')
print(f'Vector norms (first 5): {np.linalg.norm(vectors[:5], axis=1).round(4)}')

## 4. Build FAISS Index

In [ ]:
store = RecipeVectorStore()
store.build(vectors, df)
store.save()
print(f'Index size: {store.size:,} vectors')

## 5. Sample Queries — Retrieval Sanity Check

In [ ]:
def show_results(ingredients, top_k=5):
    q = embedder.encode_query(ingredients)
    results = store.search(q, top_k=top_k)
    print(f'\nQuery: {ingredients}')
    print('─' * 60)
    for r in results:
        title = r.get('clean_title') or r.get('title', 'N/A')
        score = r.get('similarity_score', 0)
        ings  = r.get('ingredients', [])[:5]
        print(f'  [{score:.3f}] {title}')
        print(f'          Ingredients: {", ".join(str(i) for i in ings)}...')

show_results(['eggs', 'flour', 'butter', 'sugar', 'milk'])
show_results(['chicken', 'garlic', 'lemon', 'olive oil', 'rosemary'])
show_results(['pasta', 'tomato', 'basil', 'mozzarella'])

## 6. Embedding Space Visualisation (t-SNE)

In [ ]:
# Visualise a random sample of 2000 recipe embeddings
sample_n = min(2000, len(vectors))
idx = np.random.choice(len(vectors), sample_n, replace=False)
sample_vecs = vectors[idx]

print('Running t-SNE (this may take ~1-2 min)...')
tsne = TSNE(n_components=2, random_state=42, perplexity=40, n_iter=500)
coords = tsne.fit_transform(sample_vecs)

plt.figure(figsize=(12, 8))
plt.scatter(coords[:, 0], coords[:, 1], alpha=0.3, s=5, c='steelblue')
plt.title(f't-SNE of {sample_n} Recipe Embeddings')
plt.axis('off')
plt.tight_layout()
plt.savefig('../data/processed/tsne_embeddings.png', dpi=150)
plt.show()

## 7. Retrieval Precision Evaluation

In [ ]:
from src.preprocessing.ingredient_parser import match_ingredients

# Evaluate: for each sampled recipe, use its own ingredients as query
# and check if the same recipe is in top-5 results (recall@5)
eval_n = min(200, len(df))
eval_df = df.sample(eval_n, random_state=42)

hits = 0
avg_ing_coverage = []

for _, row in eval_df.iterrows():
    ings = row['ingredients'] if isinstance(row['ingredients'], list) else []
    if not ings:
        continue
    q = embedder.encode_query(ings)
    results = store.search(q, top_k=5)
    result_ids = [str(r.get('id','')) for r in results]
    if str(row['id']) in result_ids:
        hits += 1
    
    # Also measure ingredient coverage of top-1 result
    if results:
        top_ings = results[0].get('ingredients', [])
        if isinstance(top_ings, list) and top_ings:
            m = match_ingredients(ings, top_ings)
            avg_ing_coverage.append(m['score'])

recall_at_5 = hits / eval_n
mean_coverage = float(np.mean(avg_ing_coverage)) if avg_ing_coverage else 0

print(f'Recall@5 (exact self-match): {recall_at_5:.2%}')
print(f'Mean ingredient coverage (top-1): {mean_coverage:.2%}')